In [1]:
import pandas as pd

In [2]:
c_e = pd.read_csv("predictions_champs_elysees.csv", sep=";", encoding="utf-8")
conv = pd.read_csv("predictions_convention.csv", sep=";", encoding="utf-8")
sts = pd.read_csv("predictions_saint_peres.csv", sep=";", encoding="utf-8")

In [3]:
df_total = pd.concat([c_e, conv, sts], ignore_index=True)

df_total = df_total.loc[:, ~df_total.columns.str.contains('^Unnamed')]

df_total['datetime'] = pd.to_datetime(df_total['datetime']) 
df_total['datetime'] = df_total['datetime'].dt.strftime("%Y-%m-%d %H:%M")
df_total.head()

,arc,datetime,debit_horaire,taux_occupation
0,Champs-Elysées,2025-11-09 00:00,855.081202,14.983612
1,Champs-Elysées,2025-11-09 01:00,865.542637,15.453246
2,Champs-Elysées,2025-11-09 02:00,817.099936,13.803854
3,Champs-Elysées,2025-11-09 03:00,781.530067,12.703053
4,Champs-Elysées,2025-11-09 04:00,715.949621,12.670676


In [4]:
from datetime import datetime

def verifier_format_df(df):
    """
    Vérifie que le DataFrame respecte le format attendu :
    - Colonnes : arc, datetime, debit_horaire, taux_occupation
    - Types des colonnes
    - Format datetime : 'YYYY-MM-DD HH:MM'
    - Valeurs autorisées dans arc
    - Nombre de lignes par arc = 360

    Lève une AssertionError si un critère n'est pas respecté.
    """

    # ✅ 1. Colonnes attendues
    colonnes_attendues = ["arc", "datetime", "debit_horaire", "taux_occupation"]
    assert list(df.columns) == colonnes_attendues, (
        f"❌ Colonnes incorrectes.\n"
        f"   Attendu : {colonnes_attendues}\n"
        f"   Trouvé  : {list(df.columns)}"
    )

    # ✅ 2. Types simples attendus
    assert df["arc"].dtype == object, "❌ 'arc' doit être de type object (string)"
    assert df["datetime"].dtype == object, "❌ 'datetime' doit rester en string (format ISO)"
    assert df["debit_horaire"].dtype in [float, "float64"], "❌ 'debit_horaire' doit être float"
    assert df["taux_occupation"].dtype in [float, "float64"], "❌ 'taux_occupation' doit être float"

    # ✅ 3. Format datetime
    try:
        df["datetime"].apply(lambda x: datetime.strptime(x, "%Y-%m-%d %H:%M"))
    except Exception as e:
        raise ValueError("❌ Format incorrect dans 'datetime'. Attendu 'YYYY-MM-DD HH:MM'") from e

    # ✅ 4. Vérification des valeurs possibles dans 'arc'
    valeurs_attendues = ["Champs-Elysées", "Convention", "Saint-Pères"]
    uniques = sorted(df["arc"].unique())
    assert uniques == sorted(valeurs_attendues), (
        f"❌ Valeurs inattendues dans 'arc'.\n"
        f"   Attendu : {valeurs_attendues}\n"
        f"   Trouvé  : {uniques}"
    )

    # ✅ 5. Nombre de lignes attendu par arc (360 ici, peut être modifié)
    for arc in valeurs_attendues:
        n = df[df["arc"] == arc].shape[0]
        assert n == 72, f"❌ L'arc '{arc}' devrait avoir 360 lignes, trouvé {n}"

    print("✅ DataFrame valide et conforme au format attendu.")
    return True



In [5]:
verifier_format_df(df_total)

✅ DataFrame valide et conforme au format attendu.


True

In [6]:
df_total.to_csv("predictions_finales.csv", sep=";" ,index=False, encoding='utf-8')